# Curriculum 04 · Lab 5 — Contextual compression: shrink context before the answer step

**Goal:** Cut the context tax. Plain top-k hands the answer LLM every
retrieved passage in full — including the sentences the question doesn't
care about. Contextual compression runs each passage through an LLM with a
"keep only what answers this question" prompt (`LLMChainExtractor`), and the
answer step reads only the surviving sentences.

```
Retriever  : ContextualCompressionRetriever (langchain-classic)
Compressor : LLMChainExtractor — one LLM call per retrieved document
             (2 questions x k=3 = 6 calls per run, on Groq)
LLM        : llama-3.3-70b-versatile (ChatGroq) — never the embedder
Embedding  : BGE (BAAI/bge-base-en-v1.5, local, CPU)
Measure    : raw vs compressed chars / whitespace-token estimate + reduction %
Data       : rag-mini-wikipedia (first 100 passages, questions 1606/1610)
```

**Why compression:** the LLM bill for the *answer* step grows with context
length, and redundant sentences actively dilute the answer. The compressor
pays a small per-document LLM cost up front (a couple of seconds each on
Groq) to keep the answer context small and clean — the raw passages stay
available if a question needs them.

This is the fifth lab of track 04-retrieval (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present),
loads `GROQ_API_KEY` from the repo-root `.env`, and puts the repo root on
`sys.path` so every repo-relative path behaves exactly like the lab script.

**WHY:** Everything embeds **locally** with BGE. The only API call in this
lab is the *compressor* LLM — Groq's `llama-3.3-70b-versatile` (a commented
Gemini alternative is kept in the source). The repo-root `.env` holds
`GROQ_API_KEY`; the imports cell resolves the repo root by walking up from
the kernel cwd and `cd`s into it.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second. The BGE model loads lazily when
the experiment cell first calls it; the Groq key is read from `.env`.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings (langchain_huggingface)
#   faiss-cpu             -> the FAISS vector store (langchain_community)
#   langchain-classic     -> ContextualCompressionRetriever + LLMChainExtractor
#   langchain-groq        -> ChatGroq (the compressor LLM)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu langchain-classic langchain-groq python-dotenv pandas



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env

from langchain_classic.retrievers import (  # noqa: E402
    ContextualCompressionRetriever,
)
from langchain_classic.retrievers.document_compressors import (  # noqa: E402
    LLMChainExtractor,
)
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_groq import ChatGroq  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
# (Gemini alternative: from langchain_google_genai import ChatGoogleGenerativeAI)


/tmp/ipykernel_1406949/396708888.py:33: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** The corpus constants plus the compression knobs:
`QUESTION_IDS = [1606, 1610]` (2 questions x `TOP_K = 3` = 6 compressor
calls per run) and `LLM_MODEL = "llama-3.3-70b-versatile"` — the Groq model
doing the compression (a commented Gemini alternative is kept in the source).

**WHY:** The lab quantifies the compression tax: every run makes exactly
`# questions x k` LLM round-trips, and the demo prints the per-question
reduction percentage so you can judge whether the tax is worth the payoff.


In [3]:
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610]  # 2 questions x K=3 = 6 compressor calls per run
TOP_K = 3
LLM_MODEL = "llama-3.3-70b-versatile"  # Groq is the *compressor* LLM, never the embedder
# (Gemini alternative: LLM_MODEL = "gemini-2.5-flash" — needs GOOGLE_API_KEY in .env)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2 · Load — corpus + questions from the fresh parquet files

**WHAT:** `load_passages` pulls the first `n` passages (text + ids) from
`passages.parquet`; `load_questions` pulls specific rows by id from
`test.parquet`; `preview` flattens a passage for one-line printing.

**WHY:** Identical helpers to the rest of the track keep the labs directly
comparable — the only thing that changes here is the retriever wrapper.


In [4]:
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


def tokens(text: str) -> int:
    """Cheap token estimate: whitespace-split word count."""
    return len(text.split())


def total_chars(docs: list[Document]) -> int:
    """Total characters across a list of documents."""
    return sum(len(d.page_content) for d in docs)


def total_tokens(docs: list[Document]) -> int:
    """Total token estimate across a list of documents."""
    return sum(tokens(d.page_content) for d in docs)


## 3 · Experiment — raw retrieval vs LLM-compressed retrieval

**WHAT:** `run_experiment` embeds the 100-passage subset once, builds the
FAISS store and a plain top-k retriever, wraps it in the
`ContextualCompressionRetriever` with an `LLMChainExtractor`, then retrieves
each question through BOTH paths — recording raw and compressed character
counts, whitespace-token estimates, and the compressor's wall time.

**WHY:** Everything the demo and gate need (raw vs compressed sizes, the
reduction, the retained-keyword check) comes from the same two retrievals —
one shared `exp`, no recomputation between the printed and the verified
numbers.


In [5]:
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed locally (BGE) and index in-memory with langchain-native FAISS -
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME, encode_kwargs={"normalize_embeddings": True}
    )
    t0 = time.perf_counter()
    store = FAISS.from_documents(chunks, embedder)
    index_s = time.perf_counter() - t0

    # --- Base retriever (raw top-k) -----------------------------------------
    base_retriever = store.as_retriever(search_kwargs={"k": TOP_K})

    # --- Compressor LLM + the wrapped retriever ------------------------------
    # Groq only compresses; every embedding above is local BGE.
    # (Gemini alternative: llm = ChatGoogleGenerativeAI(model=LLM_MODEL, temperature=0.0))
    llm = ChatGroq(model=LLM_MODEL, temperature=0.0)
    compressor = LLMChainExtractor.from_llm(llm)
    compressed_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=base_retriever
    )

    # --- Per question: raw vs compressed -------------------------------------
    results = []
    for qid, qtext in questions:
        raw_docs = base_retriever.invoke(qtext)
        t0 = time.perf_counter()
        comp_docs = compressed_retriever.invoke(qtext)
        comp_s = time.perf_counter() - t0
        results.append(
            {
                "qid": qid,
                "question": qtext,
                "raw_docs": raw_docs,
                "compressed_docs": comp_docs,
                "raw_chars": total_chars(raw_docs),
                "raw_tokens": total_tokens(raw_docs),
                "comp_chars": total_chars(comp_docs),
                "comp_tokens": total_tokens(comp_docs),
                "comp_s": comp_s,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "indexed": len(passage_texts),
        "index_s": index_s,
        "results": results,
    }


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — embedding, indexing, and 6 Groq
compressor calls (2 questions x k=3) take a few seconds. The artifact dict is
kept as `exp`.

**WHY:** As in the earlier labs, demo and gate both read this single `exp`.
The gate's keyword-retention check depends on the actual compressed output,
so the LLM calls must happen here, once.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the corpus and index summary, then per question
the raw vs compressed context side by side (chars / token estimate), the
reduction percentage and compressor time, and the raw top-1 / compressed
first-hit previews.

**WHY:** The reduction line is the headline — expect 50%+ token savings when
the extractor drops the off-topic sentences — and the previews show the
compressor kept exactly the sentences the question is about. Read the timing
line as the cost of that cleanliness.


In [7]:
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 05 — Contextual Compression: shrink context before the answer step")
    print(f"{BGE_MODEL_NAME} (local) -> FAISS top-{TOP_K} -> {LLM_MODEL} extractor")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    {len(exp['questions'])} questions from test.parquet:")
    for qid, qtext in exp["questions"]:
        print(f"      [{qid}] {qtext}")

    print(f"\n[2] Index:")
    print(f"    FAISS index built in {exp['index_s']:.3f}s over local BGE embeddings")

    print(f"\n[3] Raw vs compressed context (chars / token-estimate):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["question"]}"')
        raw_c, raw_t = r["raw_chars"], r["raw_tokens"]
        comp_c, comp_t = r["comp_chars"], r["comp_tokens"]
        print(f"      raw        {raw_c:5d} chars / {raw_t:4d} tokens  (k={len(r['raw_docs'])})")
        print(f"      compressed {comp_c:5d} chars / {comp_t:4d} tokens  (k={len(r['compressed_docs'])})")
        red_c = 100.0 * (raw_c - comp_c) / raw_c if raw_c else 0.0
        red_t = 100.0 * (raw_t - comp_t) / raw_t if raw_t else 0.0
        print(f"      reduction  {red_c:5.1f}% chars / {red_t:5.1f}% tokens "
              f"({r['comp_s']:.1f}s compressor time)")
        print("      raw top-1:  " + preview(r["raw_docs"][0].page_content))
        if r["compressed_docs"]:
            print("      compressed: " + preview(r["compressed_docs"][0].page_content))
        else:
            print("      compressed: (empty — extractor dropped every sentence)")

    print("\n[4] Takeaway")
    print("    The compressor keeps only the sentences the question is about,")
    print("    so the answer step reads a fraction of the original context.")
    print("    The price: one LLM call per retrieved document (2 questions x")
    print(f"    k={TOP_K} = {2 * TOP_K} calls here). Contextual compression trades")
    print("    that small LLM bill for a smaller, cleaner answer context —")
    print("    and the raw passages stay available if a question needs them.")


In [8]:
print_demo(exp)


Lab 05 — Contextual Compression: shrink context before the answer step
BAAI/bge-base-en-v1.5 (local) -> FAISS top-3 -> llama-3.3-70b-versatile extractor

[1] Corpus (deterministic subset, no randomness):
    100 passages (first 100 of 3200, ids 0..99)
    2 questions from test.parquet:
      [1606] Is Uruguay's capital Montevideo?
      [1610] Who founded Montevideo?

[2] Index:
    FAISS index built in 1.111s over local BGE embeddings

[3] Raw vs compressed context (chars / token-estimate):

    Q[1606] "Is Uruguay's capital Montevideo?"
      raw          823 chars /  128 tokens  (k=3)
      compressed   174 chars /   25 tokens  (k=3)
      reduction   78.9% chars /  80.5% tokens (0.9s compressor time)
      raw top-1:  Montevideo, Uruguay's capital.
      compressed: Montevideo, Uruguay's capital.

    Q[1610] "Who founded Montevideo?"
      raw         1028 chars /  161 tokens  (k=3)
      compressed   198 chars /   32 tokens  (k=2)
      reduction   80.7% chars /  80.1% tokens (0.

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate`: exactly `N_PASSAGES` indexed, every
question returning `TOP_K` raw hits, and per question — compressed context
non-empty, compressed chars strictly below raw chars, and the compressed
context still containing "montevideo" (the answer's keyword must survive
compression, not just shrink).

**WHY:** `python 05-compression.py --verify` must print 8/8 PASS; this cell
proves the notebook reproduces the verified `.py` exactly. The checks are
pinned to properties that survive LLM wording variance (the extractor keeps
*which* sentences, not exact text), so the gate is stable across runs.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Structural properties (no LLM involved).
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))
    checks.append(("each question returns TOP_K raw hits",
                   all(len(r["raw_docs"]) == TOP_K for r in exp["results"])))

    # Compression stability, pinned to properties that survive LLM wording
    # variance (the extractor keeps *which* sentences, not exact text).
    for r in exp["results"]:
        tag = f"Q{r['qid']}"

        # The compressor must return at least one sentence.
        checks.append((f"{tag} compressed context is non-empty",
                       len(r["compressed_docs"]) > 0 and r["comp_chars"] > 0))

        # Compressed context must be strictly shorter than raw (in chars) —
        # LLMChainExtractor drops irrelevant sentences, so this holds with
        # margin; a small tolerance keeps it robust to 1-2 word overrides.
        checks.append((f"{tag} compressed chars < raw chars",
                       r["comp_chars"] < r["raw_chars"]))

        # The compressed context must still carry the answer's keyword —
        # the whole point is relevance, not just shrinkage. "Montevideo"
        # appears in the gold answer of both questions and survives the
        # extractor because the queries name it explicitly.
        joined = " ".join(d.page_content for d in r["compressed_docs"]).lower()
        checks.append((f"{tag} compressed context retains 'montevideo'",
                       "montevideo" in joined))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


In [10]:
verify_gate(exp)


verification gate:
  [PASS] exactly 100 passages indexed
  [PASS] each question returns TOP_K raw hits
  [PASS] Q1606 compressed context is non-empty
  [PASS] Q1606 compressed chars < raw chars
  [PASS] Q1606 compressed context retains 'montevideo'
  [PASS] Q1610 compressed context is non-empty
  [PASS] Q1610 compressed chars < raw chars
  [PASS] Q1610 compressed context retains 'montevideo'


0